# Run the app

A runnable runbook for this project. Click ▶ next to a cell (or press **Shift+Enter** inside it) to run it
**on your machine**, from the project folder. The first run asks you to trust the notebook — it comes from the
repo, so review the cells first.

Every cell sees the active environment's variables as `DEVALL_<name>` (here `DEVALL_baseUrl`); secrets are
only passed to cells you explicitly allow (cell menu → *Give this cell the environment's secrets*).

## 1. Run every request (shell)

The same collection run CI uses. Needs the `devall` CLI on your PATH
(`cargo install --path crates/devall-cli` from the /dev/all repo). Requests under `posts/` send
`{{secret.API_TOKEN}}`; the CLI reads it from your keychain, or from `DEVALL_SECRET_API_TOKEN` when you give
this cell the environment's secrets.

In [ ]:
if command -v devall >/dev/null 2>&1; then
  devall run . --env "${DEVALL_ENV:-public}"
else
  echo "The devall CLI isn't on your PATH yet."
  echo "Build it from the /dev/all repo:  cargo install --path crates/devall-cli"
  exit 1
fi

## 2. Call the API from a shell

`$DEVALL_baseUrl` comes from the active environment (*Public API*). On Windows, shell cells run in Git Bash.

In [ ]:
base="${DEVALL_baseUrl:-https://jsonplaceholder.typicode.com}"
echo "GET $base/users/1"
curl -s "$base/users/1" | head -c 400
echo

## 3. PowerShell

Runs with PowerShell 7 when installed, else Windows PowerShell.

In [ ]:
$base = if ($env:DEVALL_baseUrl) { $env:DEVALL_baseUrl } else { 'https://jsonplaceholder.typicode.com' }
$user = Invoke-RestMethod "$base/users/1"
'{0} <{1}> works at {2}' -f $user.name, $user.email, $user.company.name

## 4. JavaScript (Node.js)

One Node.js process per notebook: top-level `await` works and variables survive between cells. The value of
the last expression is shown below the cell.

In [ ]:
const base = process.env.DEVALL_baseUrl ?? 'https://jsonplaceholder.typicode.com';
const posts = await (await fetch(`${base}/posts?userId=1`)).json();
console.log(`${posts.length} posts by user 1`);
posts.slice(0, 3).map((p) => p.title)

In [ ]:
// `posts` is still defined: same kernel as the previous cell.
posts.filter((p) => p.body.includes('quia')).length

## 5. Python

One Python process per notebook, standard library only. The first cell defines `todos`; the second one uses it.

In [ ]:
import json, os, urllib.request

base = os.environ.get("DEVALL_baseUrl", "https://jsonplaceholder.typicode.com")
with urllib.request.urlopen(f"{base}/todos?userId=1") as r:
    todos = json.load(r)
len(todos)

In [ ]:
done = [t for t in todos if t["completed"]]
f"{len(done)} of {len(todos)} todos are done"

## 6. `{{variables}}` in code

This cell opts in to variable substitution (`metadata.devall.templating`), so `{{baseUrl}}` and dynamic
values like `{{$uuid}}` are replaced before it runs.

In [ ]:
echo "Request id {{$uuid}} → {{baseUrl}}/users"